## NFB GLM

In [ ]:
import os 
import shutil
import numpy as np
import nibabel as nib 
import pandas as pd 
from nilearn.image import mean_img
from nilearn.glm.first_level import FirstLevelModel
from nilearn.plotting import plot_img, plot_roi, plot_design_matrix, plot_contrast_matrix

In [ ]:
input_nifti_path = "/lab-share/Neuro-Cohen-e2/Groups/IRB-P00049401/mw_motion_pipeline/runs_and_data/participant_runs/MOTION_INJECTED_FROM_ORIGINAL_IMAGE_p005-ses-04-postrifg_outputs/fmriprep_outputs_sub-005-2026-02-27_10-41-10/ses-04/func/sub-005_ses-04_task-postRIFG_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz"
input_nifti_mask = "/lab-share/Neuro-Cohen-e2/Groups/IRB-P00049401/mw_motion_pipeline/runs_and_data/participant_runs/MOTION_INJECTED_FROM_ORIGINAL_IMAGE_p005-ses-04-postrifg_outputs/p005-ses-04-postrifg_func-bold_task-postRIFG_20250826105831_36_MOTION_INJECTED_FROM_ORIGINAL.nii_bgremoved.nii.gz"
data_dictionary_path = "/lab-share/Neuro-Cohen-e2/Groups/IRB-P00049401/subject_data/p005/ses-04/realtime_logs_and_events/nfb_logs/p005_data_dictionary_20250826_11h52m42s.csv"

output_dir = "/lab-share/Neuro-Cohen-e2/Groups/IRB-P00049401/mw_motion_pipeline/runs_and_data/participant_runs/MOTION_INJECTED_FROM_ORIGINAL_IMAGE_p005-ses-04-postrifg_outputs"
os.makedirs(output_dir, exist_ok=True)

tr  = 1.06 # must be in s
sequence = '04_000024' # extract DICOMS with this sequence

In [ ]:
shutil.copy(
    input_nifti_path,
    os.path.join(output_dir, os.path.basename(input_nifti_path))
)

In [ ]:
# make mask from bg image
img = nib.load(input_nifti_mask)
data_3d = img.get_fdata()[:, :, :, 0]
mask = (data_3d > 0).astype(np.uint8)
mask_img = nib.Nifti1Image(mask, affine=img.affine, header=img.header)
input_nifti_mask = os.path.join(output_dir, "mask_img.nii.gz")
nib.save(mask_img, input_nifti_mask)

In [ ]:
nifti_image: nib.Nifti1Image = nib.load(input_nifti_path)
mean_nifti_inmage: nib.Nifti1Image = mean_img(nifti_image)

mean_image_plot_path = os.path.join(output_dir, "mean_input_nifti_image.png")
plot_img(
    mean_nifti_inmage, 
    title=f"{os.path.basename(input_nifti_path)}\n(Mean Image)",
    output_file=mean_image_plot_path)
plot_img(
    mean_nifti_inmage, 
    title=f"{os.path.basename(input_nifti_path)}\n(Mean Image)")

plot_img(
    mask_img, 
    title=f"Mask Image")

In [ ]:

def get_events(data_dictionary_csv, tr):
    
    events = {
        "trial_type": [],
        "onset": [],
        "duration": []
    }
    
    with open(file=data_dictionary_csv, mode='r') as file:
        all_lines = file.readlines()
        for i, line in enumerate(all_lines):
            if 'dicom_path' and sequence in line:
                for line in all_lines[i:]:
                    if f'{str(tr)}"' in line: 
                        trial_type = line.split()[1].strip()
                        duration = float(line.split()[3].replace('"', ''))
                        onset  = 0
                        if len(events['onset']) > 0:
                            onset  = round(events['onset'][-1] + duration, 2)
                        events['trial_type'].append(trial_type)
                        events['onset'].append(onset)
                        events['duration'].append(duration)
                        break
    
    return events


events = pd.DataFrame(get_events(
    data_dictionary_csv=data_dictionary_path,
    tr=tr
))

print("Event Csv:")
events

In [ ]:
if nifti_image.shape[3] != len(events):
    print(f"WARNING: The number of volumes in the inputted image ({nifti_image.shape[3]}) does not equal the number of rows in the event csv ({len(events)})")


In [ ]:
fmri_glm = FirstLevelModel(
    mask_img=input_nifti_mask,
    t_r=tr,
    noise_model="ar1",
    standardize=True,
    hrf_model="spm + derivative",
    drift_model="cosine",
    high_pass=0.01,
    minimize_memory=False,
    smoothing_fwhm=7.2,
    n_jobs=1
)



In [ ]:
fmri_glm.fit(
    run_imgs=nifti_image,
    events=events
)

In [ ]:
print(f"Info On Mask Image:")
print(fmri_glm.masker_.mask_img_)
plot_roi(fmri_glm.masker_.mask_img_, title="Mask used in GLM")

In [ ]:
design_matrix = fmri_glm.design_matrices_[0]
design_matrix_plot_path = os.path.join(output_dir, "design_matrix.png")
plot_design_matrix(design_matrix, output_file=design_matrix_plot_path)
plot_design_matrix(design_matrix)

In [ ]:
n_regressors = design_matrix.shape[1]
activation = np.zeros(n_regressors)
activation[0] = 1
activation[2] = -1
print(f"Contrasts: {activation}")

contrast_plot_path = os.path.join(output_dir, "contrast_matrix.png")
plot_contrast_matrix(
    contrast_def=activation, 
    design_matrix=design_matrix, 
    output_file=contrast_plot_path
)
plot_contrast_matrix(
    contrast_def=activation, 
    design_matrix=design_matrix
)

In [ ]:
z_maps = fmri_glm.compute_contrast(
    contrast_def=activation,
    output_type="all"
)

for stat_name, map_data in z_maps.items():
    output_path = os.path.join(output_dir, f"{stat_name}.png")
    plot_img(
        map_data,
        bg_img=mean_nifti_inmage,
        display_mode='ortho',
        colorbar=True,
        title=f"NFB > Rest {stat_name} Map for Sequence {sequence}",
        output_file=output_path)
    plot_img(
        map_data,
        bg_img=mean_nifti_inmage,
        display_mode='ortho',
        colorbar=True,
        title=f"NFB > Rest {stat_name} Map for Sequence {sequence}")
    nib.save(map_data, os.path.join(output_dir, f"{stat_name}.nii.gz"))

In [ ]:
shutil.copy(
    "NFB_GLM.ipynb",
    os.path.join(output_dir, os.path.basename("NFB_GLM.ipynb"))
)